In [ ]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-1227"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)
N_BOOT = 200 + 10 * SEED_DIGIT
MIN_OBS = 10

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Bootstrap Repetitions:", N_BOOT)

Student: Solis TUPM-26-1227
Seed Digit: 7
ID Sum: 20
Random Seed: 261227
Bootstrap Repetitions: 270


In [2]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("Dataset files verified.")

Workspace: C:\Users\n0n3\Lab7_NumPy_Pandas_Solis\Lab7_NumPy_Pandas_Additional_Exercises
Dataset files verified.


In [3]:
# [CELL 3] LOAD DATA AND IDENTIFY USABLE FEATURES
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.shape[0] == len(labels)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_mask = ~(all_missing | constant)
usable_indices = np.flatnonzero(usable_mask)

pass_mask = labels == -1
fail_mask = labels == 1

print("Shape:", raw_measurements.shape)
print("Pass:", pass_mask.sum(), "| Fail:", fail_mask.sum())
print("Usable Features:", len(usable_indices))

Shape: (1567, 590)
Pass: 1463 | Fail: 104
Usable Features: 474


In [4]:
# [CELL 4] REPRODUCIBLE SELECTION OF FIVE FEATURES (>= 10 observed per outcome group)
obs_pass = (~missing[pass_mask]).sum(axis=0)
obs_fail = (~missing[fail_mask]).sum(axis=0)

eligible = np.flatnonzero(usable_mask & (obs_pass >= MIN_OBS) & (obs_fail >= MIN_OBS))

rng_select = np.random.default_rng(RANDOM_SEED)
selected_indices = np.sort(rng_select.choice(eligible, size=5, replace=False))
selected_names = feature_names[selected_indices].tolist()
k = len(selected_indices)

selected = raw_measurements[:, selected_indices]

print("Eligible Features:", len(eligible))
print("Selected Features:", selected_names)
print("Observed Pass:", obs_pass[selected_indices])
print("Observed Fail:", obs_fail[selected_indices])

Eligible Features: 470
Selected Features: ['Feature_019', 'Feature_020', 'Feature_050', 'Feature_075', 'Feature_204']
Observed Pass: [1454 1463 1462 1439 1457]
Observed Fail: [103 104 104 104 104]


In [5]:
# [CELL 5] TWO PREPARATION STRATEGIES
# Strategy 1 - available-case: drop missing values separately for each feature
ac_pass = [selected[pass_mask, j][~np.isnan(selected[pass_mask, j])] for j in range(k)]
ac_fail = [selected[fail_mask, j][~np.isnan(selected[fail_mask, j])] for j in range(k)]

# Strategy 2 - median imputation using medians from the FULL dataset
full_medians = np.nanmedian(selected, axis=0)
imputed = np.where(np.isnan(selected), full_medians, selected)
mi_pass, mi_fail = imputed[pass_mask], imputed[fail_mask]

print("Available-Case n (Pass):", [len(a) for a in ac_pass])
print("Available-Case n (Fail):", [len(a) for a in ac_fail])
print("Imputed n (Pass/Fail):", mi_pass.shape[0], "/", mi_fail.shape[0])
print("Missing After Imputation:", int(np.isnan(imputed).sum()))

Available-Case n (Pass): [1454, 1463, 1462, 1439, 1457]
Available-Case n (Fail): [103, 104, 104, 104, 104]
Imputed n (Pass/Fail): 1463 / 104
Missing After Imputation: 0


In [6]:
# [CELL 6] OBSERVED MEAN DIFFERENCES (Fail - Pass)
ac_diff = np.array([ac_fail[j].mean() - ac_pass[j].mean() for j in range(k)])
mi_diff = mi_fail.mean(axis=0) - mi_pass.mean(axis=0)

print("Available-Case Difference:", ac_diff)
print("Median-Imputed Difference:", mi_diff)

Available-Case Difference: [ 0.0269  0.0016 -0.5266  0.0025 68.6934]
Median-Imputed Difference: [ 0.0267  0.0016 -0.526   0.0025 68.8061]


In [7]:
# [CELL 7] STRATIFIED BOOTSTRAP (independent resampling inside each outcome group)
def bootstrap_differences(seed, n_boot):
    rng = np.random.default_rng(seed)

    # Median imputation: resample records within each outcome group
    ip = rng.integers(0, mi_pass.shape[0], size=(n_boot, mi_pass.shape[0]))
    jf = rng.integers(0, mi_fail.shape[0], size=(n_boot, mi_fail.shape[0]))
    mi = mi_fail[jf].mean(axis=1) - mi_pass[ip].mean(axis=1)      # (n_boot, k)

    # Available-case: resample observed values within each group and feature
    ac = np.empty((n_boot, k))
    for j in range(k):
        p, f = ac_pass[j], ac_fail[j]
        ac[:, j] = (f[rng.integers(0, len(f), size=(n_boot, len(f)))].mean(axis=1)
                    - p[rng.integers(0, len(p), size=(n_boot, len(p)))].mean(axis=1))
    return ac, mi

ac_boot, mi_boot = bootstrap_differences(RANDOM_SEED, N_BOOT)

ac_boot_mean, mi_boot_mean = ac_boot.mean(axis=0), mi_boot.mean(axis=0)
ac_ci = np.percentile(ac_boot, [2.5, 97.5], axis=0)
mi_ci = np.percentile(mi_boot, [2.5, 97.5], axis=0)

print("Bootstrap Repetitions:", N_BOOT)
print("Available-Case Boot Mean:", ac_boot_mean)
print("Median-Imputed Boot Mean:", mi_boot_mean)
print("Available-Case 95% CI Low/High:\n", ac_ci)
print("Median-Imputed 95% CI Low/High:\n", mi_ci)

Bootstrap Repetitions: 270
Available-Case Boot Mean: [ 0.0272  0.0015 -0.6147  0.0025 71.4409]
Median-Imputed Boot Mean: [ 0.0273  0.0017 -0.4939  0.0024 78.0176]
Available-Case 95% CI Low/High:
 [[  0.0127  -0.0016  -1.9621  -0.003  -48.0231]
 [  0.0451   0.0046   1.0532   0.0087 276.5606]]
Median-Imputed 95% CI Low/High:
 [[  0.0121  -0.001   -1.9953  -0.0033 -47.8281]
 [  0.047    0.0045   1.0093   0.0099 283.4462]]


In [8]:
# [CELL 8] COMPARE DIRECTION, INTERVALS, AND RANKINGS
def rank_by_abs(diff):
    order = np.argsort(-np.abs(diff), kind="stable")
    ranks = np.empty(len(diff), dtype=int)
    ranks[order] = np.arange(1, len(diff) + 1)
    return ranks

ac_rank, mi_rank = rank_by_abs(ac_diff), rank_by_abs(mi_diff)

sign_changed = np.sign(ac_diff) != np.sign(mi_diff)
rank_changed = ac_rank != mi_rank
diff_change = mi_diff - ac_diff

ac_excl0 = (ac_ci[0] > 0) | (ac_ci[1] < 0)      # CI excludes zero
mi_excl0 = (mi_ci[0] > 0) | (mi_ci[1] < 0)

big = int(np.argmax(np.abs(diff_change)))

print("Sign Changed:", sign_changed)
print("Rank Changed:", rank_changed)
print("Change in Difference:", diff_change)
print("AC CI Excludes 0:", ac_excl0)
print("MI CI Excludes 0:", mi_excl0)

Sign Changed: [False False False False False]
Rank Changed: [False False False False False]
Change in Difference: [-0.0002  0.      0.0006 -0.      0.1127]
AC CI Excludes 0: [ True False False False False]
MI CI Excludes 0: [ True False False False False]


In [9]:
# [CELL 9] EXPORT robustness_comparison.csv
comparison = pd.DataFrame({
    "Feature": selected_names,
    "AC_Pass_N": [len(a) for a in ac_pass], "AC_Fail_N": [len(a) for a in ac_fail],
    "AC_Diff": ac_diff, "AC_Boot_Mean": ac_boot_mean,
    "AC_CI_Low": ac_ci[0], "AC_CI_High": ac_ci[1],
    "AC_CI_Excludes_Zero": ac_excl0, "AC_Rank": ac_rank,
    "MI_Diff": mi_diff, "MI_Boot_Mean": mi_boot_mean,
    "MI_CI_Low": mi_ci[0], "MI_CI_High": mi_ci[1],
    "MI_CI_Excludes_Zero": mi_excl0, "MI_Rank": mi_rank,
    "Diff_Change": diff_change,
    "Sign_Changed": sign_changed, "Rank_Changed": rank_changed,
})

comparison_path = OUTPUT_DIR / "robustness_comparison.csv"
report_path = OUTPUT_DIR / "sensitivity_report.txt"
comparison.to_csv(comparison_path, index=False)

print(comparison[["Feature", "AC_Diff", "MI_Diff", "AC_Rank", "MI_Rank"]]
      .round(4).to_string(index=False))

    Feature  AC_Diff  MI_Diff  AC_Rank  MI_Rank
Feature_019   0.0269   0.0267        3        3
Feature_020   0.0016   0.0016        5        5
Feature_050  -0.5266  -0.5260        2        2
Feature_075   0.0025   0.0025        4        4
Feature_204  68.6934  68.8061        1        1


In [10]:
# [CELL 10] REPRODUCIBILITY CHECK (same seed, run again)
ac_boot2, mi_boot2 = bootstrap_differences(RANDOM_SEED, N_BOOT)
repeat_ok = np.array_equal(ac_boot, ac_boot2) and np.array_equal(mi_boot, mi_boot2)

# Same selection again from the same seed
reselect = np.sort(np.random.default_rng(RANDOM_SEED).choice(eligible, size=5, replace=False))
select_ok = np.array_equal(reselect, selected_indices)

print("Bootstrap Repeats Identically:", bool(repeat_ok))
print("Feature Selection Repeats Identically:", bool(select_ok))

Bootstrap Repeats Identically: True
Feature Selection Repeats Identically: True


In [11]:
# [CELL 11] WRITE sensitivity_report.txt AND VERIFY EXPORTS
def fmt_ids(mask):
    return f"{int(mask.sum())} ({', '.join(str(i) for i in selected_indices[mask])})" if mask.any() else "0"

lines = [
    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT",
    f"Student: {LAST_NAME} | Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED} | Bootstrap Repetitions: {N_BOOT}",
    f"Selected Features: {', '.join(selected_names)}",
    "Difference = mean(Fail) - mean(Pass); strategies: available-case (AC) and median imputation (MI).",
    "",
    "Feature results:",
]
for j, name in enumerate(selected_names):
    lines.append(
        f"- {name}: AC diff {ac_diff[j]:.4f} [{ac_ci[0, j]:.4f}, {ac_ci[1, j]:.4f}] rank {ac_rank[j]} | "
        f"MI diff {mi_diff[j]:.4f} [{mi_ci[0, j]:.4f}, {mi_ci[1, j]:.4f}] rank {mi_rank[j]}"
    )

lines += [
    "",
    "Interpretation:",
    f"- Features whose difference sign changed between strategies: {fmt_ids(sign_changed)}.",
    f"- Features whose ranking changed between strategies: {fmt_ids(rank_changed)}.",
    f"- Largest change in mean difference: {selected_names[big]} ({abs(diff_change[big]):.4f}).",
    f"- Intervals excluding zero: AC {fmt_ids(ac_excl0)}; MI {fmt_ids(mi_excl0)}.",
    f"- Reproducibility (same seed, repeated run): {'PASSED' if repeat_ok and select_ok else 'FAILED'}.",
    "",
    "Notes:",
    "- Intervals are percentile 95% bootstrap intervals from stratified resampling with replacement.",
    "- Median imputation uses full-dataset medians and keeps every record; available-case uses fewer values per feature.",
    "- A CI that includes zero means the Pass/Fail difference is not clearly separated from zero for that strategy.",
    "- Mean differences are in raw feature units, so rankings by size reflect feature scale as well as effect.",
    "- Group differences describe association and do not establish failure causes.",
]
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

loaded = pd.read_csv(comparison_path)
csv_ok = (len(loaded) == k
        and np.allclose(loaded["AC_Diff"], ac_diff) and np.allclose(loaded["MI_CI_High"], mi_ci[1])
        and loaded["Rank_Changed"].sum() == rank_changed.sum())
report_ok = report_path.is_file() and "PASSED" in report_path.read_text(encoding="utf-8")

verified = bool(repeat_ok and select_ok and csv_ok and report_ok)

print("CSV Reload Verified:", bool(csv_ok))
print("Report Created:", report_ok)

CSV Reload Verified: True
Report Created: True


In [12]:
# [CELL 12] ASSESSMENT DATA (one line per answer)
def ids(mask):
    return f"{int(mask.sum())} ({','.join(str(i) for i in selected_indices[mask])})" if mask.any() else "0"

print("Selected Features:", ", ".join(str(i) for i in selected_indices))
print("Bootstrap Repetitions:", N_BOOT)
print("Largest Change in Mean Difference:",
    f"{abs(diff_change[big]):.4f} ({selected_indices[big]})")
print("Features with Changed Difference Sign:", ids(sign_changed))
print("Features with Changed Ranking:", ids(rank_changed))
print("Reproducibility Verified:", "Yes" if verified else "No")

Selected Features: 19, 20, 50, 75, 204
Bootstrap Repetitions: 270
Largest Change in Mean Difference: 0.1127 (204)
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: Yes
